# NB_ENR_POI_REVIEW

APPEND `lh_vv_gold.enr.gld_enr_poi_review`. Chỉ 3rd-party.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
def _flag_keys(ctx, role, key, scope):
    # D-04: cờ chỉ đổi khi tập khoá landing đổi; không phụ thuộc grain (D-03).
    # Không có baseline (cạnh mới, FULL, FORCE, mất version) -> mọi khoá trong scope: lần đầu sau patch ghi lại hết.
    ctx.changed(role)  # gọi trước: có thể đổi basis sang FULL_VERSION_UNAVAILABLE
    info = ctx.basis[role]
    if info["basis"] not in ("EDGE", "MARKER") or info["from"] is None:
        return scope
    if info["from"] >= info["to"]:
        return scope.limit(0)
    now = ctx._input_at(role, info["to"]).select(key).where(F.col(key).isNotNull()).distinct()
    old = ctx._input_at(role, info["from"]).select(key).where(F.col(key).isNotNull()).distinct()
    return now.subtract(old).union(old.subtract(now))


def build(ctx: NodeContext) -> DataFrame:
    live_ids = ctx.input("review").select("review_id").where(F.col("review_id").isNotNull()).distinct()
    changed_ids = (ctx.changed("review").select("review_id").where(F.col("review_id").isNotNull())
                   .union(_flag_keys(ctx, "multi_lang", "review_id", live_ids))
                   .distinct()).persist(StorageLevel.MEMORY_AND_DISK)
    ctx.cached.append(changed_ids)
    ml_ids = ctx.input("multi_lang").select("review_id").where(F.col("review_id").isNotNull()).distinct()
    ctx.warn("multi_lang_key_not_in_review", ml_ids.join(live_ids, "review_id", "left_anti").count())
    ml = ml_ids.join(changed_ids, "review_id", "left_semi").withColumn("ml_hit", F.lit(True))
    rv = ctx.input("review").join(changed_ids, "review_id", "left_semi").join(ml, "review_id", "left")
    cast_rating = F.col("rating").cast("decimal(3,1)")
    overflow = rv.where(cast_rating.isNull() & F.col("rating").isNotNull()).count()
    if overflow:
        raise QualityGateError(f"rating_cast_overflow: {overflow}")
    review_date = iso_utc_ts(F.col("time"))
    mismatch = rv.where(~(review_date.eqNullSafe(F.col("published_at")))).count()
    ctx.warn("review_date_ne_published_at", mismatch)
    return rv.select(
        F.col("review_id"),  # COL-022 — MQ-062, MQ-039, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("poi_id"),  # COL-023 — MQ-045, MQ-062, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        cast_rating.alias("rating"),  # COL-025 — MQ-051, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("text").alias("review_text"),  # COL-027 — MQ-057, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        review_date.alias("review_date"),  # COL-031 — MQ-023, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        iso_utc_ts(F.col("time")).alias("review_updated_at"),  # COL-032 — MQ-004, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("ml_hit").isNotNull().alias("is_enriched_multi_lang"),  # D-04
        # COL-033 — MQ-052, MQ-058. created_at do lib gán.
        # COL-034 — MQ-039, MQ-052, MQ-058. updated_at do lib gán.
    )


SPEC = NodeSpec(
    trg_tbl="gld_enr_poi_review",
    inputs={"review": "slv_3p_poi_review", "multi_lang": "slv_enr_poi_review_multi_lang"},
    pk=("review_id",),
    build=build,
    required_inputs=("review",),
    block_checks=(("enrich_flag_null", "is_enriched_multi_lang IS NULL"),),
    write_mode="APPEND",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
